# Learning factors for stock market returns prediction

## Overview

**Challenge:** ENS Challenge Data 2022 (Qube Research & Technologies). Given three years of daily returns for 50 anonymous stocks, predict the next-day cross-sectional return vector. The submission is a pair $(A,\beta)$ where $A \in \mathbb{R}^{250 \times 10}$ is a Stiefel matrix ($A^\top A = I_{10}$) and $\beta \in \mathbb{R}^{10}$: the prediction for stock $i$ at date $t$ is $\hat{y}_{i,t} = [\mathbf{X}_\text{reshape}\,A\,\beta]_{(t,i)}$, with $\mathbf{X}_\text{reshape}$ containing the 250-day lag window for every (date, stock) pair. Performance is the mean cosine overlap between predicted and realised return vectors.

**Approaches explored:**

| Method | In-sample overlap |
|---|:---:|
| AR(10) — autoregressive baseline | 0.0240 |
| 5-day return + momentum (2-factor) | 0.0195 |
| Fourier bases (10 frequencies) | 0.0301 |
| PCA on $X_\text{reshape}^\top X_\text{reshape}$ | 0.0387 |
| Random search benchmark (QRT) | 0.0458 |
| **Alternating Stiefel optimisation** | **0.1332** |

**Key finding:** Alternating projected gradient descent on the Stiefel manifold — PCA-initialised, $\beta$-step via closed-form OLS, $A$-step via Euclidean gradient + SVD retraction — achieves an in-sample overlap of **0.1332**, nearly 3× the random-search benchmark. The train/validation split in the *Additional Work* section quantifies the degree of in-sample overfitting.

**Submission:** The public benchmark score is 0.0354. My score is 0.0821.

## Data preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path = './' 

X_train = pd.read_csv(path+'X_train_YG7NZSq.csv', index_col=0, sep=',')
X_train.columns.name = 'date'

Y_train = pd.read_csv(path+'Y_train_wz11VM6.csv', index_col=0, sep=',')
Y_train.columns.name = 'date'

##### A useful data representation:

To speed up upcoming computations, we reshape the data  into a DataFrame with index (date, stock) and columns the lagged return values of the stock at time "date-1, ..., date-250".

In [ ]:
X_train_reshape = pd.concat([ X_train.T.shift(i+1).stack() for i in range(250) ], axis=1).dropna()
X_train_reshape.columns = pd.Index(range(1,251), name='timeLag')

# Utilities

In [ ]:
def checkOrthonormality(A): 
    
    bool = True
    D, F = A.shape   
    Error = pd.DataFrame(A.T @ A - np.eye(F)).abs()
    
    if any(Error.unstack() > 1e-6):
        bool = False
     
    return bool

def fitBeta(A, ld=0):
    
    predictors = X_train_reshape @ A # the dataframe of the 10 factors created from A with the (date, stock) in index
    targets = Y_train.T.stack()
    s = predictors.T.shape[0]
    beta = np.linalg.inv(predictors.T @ predictors + ld * np.eye(s)) @ predictors.T @ targets
    
    return beta.to_numpy()

In [ ]:
def metric_train(A, beta): 
    
    if not checkOrthonormality(A):
        return -1.0, None 
    
    Ypred = (X_train_reshape @ A @ beta).unstack().T         
    Ytrue = Y_train
    
    Ytrue = Ytrue.div(np.sqrt((Ytrue**2).sum()), 1)    
    Ypred = Ypred.div(np.sqrt((Ypred**2).sum()), 1)

    meanOverlap = (Ytrue * Ypred).sum().mean()

    return  meanOverlap, Ypred 

# Examples (Provided by QRT)
##### Test 1: The autoregressive model AR(10)
$$
S_{t+1} := \sum_{\ell=1}^{10} \beta_\ell R_{t+1-\ell}
$$
where the $\beta_\ell$'s are fitted by minimizing the mean square prediction error on the training data set. 

In [ ]:
def autoRegA(D=250, F=10):
    
    A = np.zeros((D,F))
    for i in range(F): 
        A[i,i] = 1
    
    return A

In [ ]:
A = autoRegA()
beta = fitBeta(A)

score, Ypred = metric_train(A, beta) # public metric: 0.01282
print(score)

##### Test 2: The two factor model using '5-day returns' and 'momentum'
This model is suggested in the description of the challenge and reads
$$
S_{t+1} := \beta_1 \,R_t^{(5)} + \beta_2 \,R_{t-20}^{(230)},\qquad \text{ with }\quad R_t^{(m)}:= \frac1{\sqrt{m}}\sum_{k=1}^{m} R_{t+1-k},
$$
where we find the parameters $\beta_1$ and $\beta_2$ by minimizing the mean square prediction error on the training data set.  

*NB: the construction below actually shows how a model with $F\leq 10$ factors be recasted into the framework of the challenge.*

In [ ]:
# Step 1: Create a 250x10 matrix A with the two first columns representing the factors of interest

A = np.zeros((250,10))

A[0:5, 0] = 1/np.sqrt(5) # 5-day return factor
A[20:250, 1] = 1/np.sqrt(230) # momentum factor
# Step 2: Fill the remaining columns of A with random orthonormal vectors, that are orthogonal to the two first columns

orthoProj = np.eye(250) - np.outer(A[:, 0], A[:, 0]) - np.outer(A[:, 1], A[:, 1]) # projection matrix on the orthogonal to the span of A[:,0] and A[:,1]
A_remaining_columns = orthoProj @ np.random.randn(250, 8) # sample random vectors in the space orthogonal to the first two columns of A
A_remaining_columns = np.linalg.qr(A_remaining_columns)[0] # orthonormalize these vectors with Gram-Schmidt algorithm
A[:, 2:] = A_remaining_columns

# Step 3: Compute the mean square optimal beta_1, beta_2 and then complete the vector beta with zeros

predictors = X_train_reshape @ A[:, :2]
targets = Y_train.T.stack()
beta = np.linalg.inv(predictors.T @ predictors) @ predictors.T @ targets
beta = np.hstack([beta, np.zeros(8)])


score, Ypred = metric_train(A, beta) # public metric: 0.01787
print(score)

# My Test

##### Test 3: Let $A$ be the first 10 Fourier bases

The result is better than simple AR

In [ ]:
A = np.zeros((250,10))

# build the matrix
A[-1, -1] = 1
A[:249, 0] = 1 / np.sqrt(249)
arr = np.arange(249) / 249
# A[:249, 0] = np.cos(2*np.pi*arr*25) / np.sqrt(249/2)
ks = [10, 20, 30, 40]   # the "wave vectors"
for i in range(1, 5):
    A[:249, 2*i-1] = np.cos(2*np.pi*arr*ks[i-1]) / np.sqrt(249/2)
    A[:249, 2*i] = np.sin(2*np.pi*arr*ks[i-1]) / np.sqrt(249/2)

In [ ]:
beta = fitBeta(A)
score, Ypred = metric_train(A, beta) # public metric: 0.01282
print(score)    

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 4), sharex=True)
axes = axes.flatten()
lag_index = np.arange(1, 251)

for f, ax in enumerate(axes):
    ax.plot(lag_index, A[:, f], lw=0.9, color=f'C{f}')
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.set_title(f'Factor {f + 1}', fontsize=9)
    ax.tick_params(labelsize=7)
    if f >= 5:
        ax.set_xlabel('lag (days)', fontsize=8)

fig.suptitle(r'$A$ — Designed learning factor loadings vs. lag index (full-data training)', fontsize=11)
plt.tight_layout()
plt.show()

##### Test 4: Building $\mathbf{A}$ to be the largest $F$ eigenvectors of $\mathbf{X}^T\mathbf{X}$ (i.e. PCA)

An even better result

In [ ]:
# Build the Gram matrix X_train_reshape^T @ X_train_reshape — shape (D=250, D=250).
# Its eigenvectors are the principal directions of the lag-feature space (PCA on X_train_reshape).
M = X_train_reshape.values.T @ X_train_reshape.values

# eigh returns eigenvalues in ascending order for symmetric matrices;
# the last F columns are the F largest eigenvectors — already orthonormal.
eigenvalues, eigenvectors = np.linalg.eigh(M)
A = eigenvectors[:, -10:]     # shape (250, F)

beta = fitBeta(A)
score, Ypred = metric_train(A, beta)
print(score)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 4), sharex=True)
axes = axes.flatten()
lag_index = np.arange(1, 251)

for f, ax in enumerate(axes):
    ax.plot(lag_index, A[:, f], lw=0.9, color=f'C{f}')
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.set_title(f'Factor {f + 1}', fontsize=9)
    ax.tick_params(labelsize=7)
    if f >= 5:
        ax.set_xlabel('lag (days)', fontsize=8)

fig.suptitle(r'$A$ — Designed learning factor loadings vs. lag index (full-data training)', fontsize=11)
plt.tight_layout()
plt.show()

# A Better way to find $\mathbf{A}$

##### Alternating optimization of $(A, \beta)$ on the Stiefel manifold

Minimize the MSE loss $\mathcal{L}(A,\beta) = \|\mathbf{X}_\text{reshape}\,A\,\beta - \mathbf{y}\|^2$ by alternating:

1. **$\beta$-step (fixed $A$):** closed-form OLS — same as `fitBeta(A)`.
2. **$A$-step (fixed $\beta$):** projected gradient descent — Euclidean gradient $\nabla_A\mathcal{L} = 2\,X^\top r\,\beta^\top$ (residual $r = XA\beta - y$), gradient step, then retract onto Stiefel via SVD: $A \leftarrow UV^\top$.

**Convergence:** $\|A_\text{new} - A_\text{old}\|_F < \texttt{res}$ or `max_iter` reached.

In [ ]:
def BetaA_search(A_init, max_iter=200, res=1e-6, lr=1e-4, ld_beta=0, ld_A=0, verbose=True):
    """
    lr: learning rate
    ld: ridge factor
    """
    X = X_train_reshape.values       # (n_samples, D=250)
    y = Y_train.T.stack().values     # (n_samples,)

    A = A_init.copy()
    beta = fitBeta(A)                # warm-start beta
    scores = []

    for it in range(max_iter):
        A_old = A.copy()

        # ── A step: gradient descent + projection ─────────────────────
        r = X @ A @ beta - y                    # residuals  (n_samples,)
        grad = np.outer(X.T @ r, beta) + ld_A * A     # Euclidean gradient  (D, F)
        U, _, Vt = np.linalg.svd(A - lr * grad, full_matrices=False)
        A = U @ Vt                                 # re-project onto Stiefel manifold

        # ── beta step ─────────────
        beta = fitBeta(A, ld=ld_beta)

        # ── logging and convergence check ────────────
        diff  = np.linalg.norm(A - A_old, 'fro')
        score, _ = metric_train(A, beta)
        scores.append(score)

        if verbose:
            if (it + 1) % 20 == 0:
                print(f'iter {it+1:4d} | overlap = {score:.5f} | ||ΔA||_2 = {diff:.2e}')

        if diff < res:
            print(f'Converged at iteration {it + 1}  (||ΔA||_2 = {diff:.2e} < {res:.0e})')
            break
    else:
        print(f'Reached max_iter={max_iter}  (final overlap = {scores[-1]:.5f})')

    return A, beta, scores

In [ ]:
# initialization
M = X_train_reshape.values.T @ X_train_reshape.values
eigenvalues, eigenvectors = np.linalg.eigh(M)
A_init = eigenvectors[:, -10:]     # shape (250, 10), the PCA decomposition as initial guess

# optimize
A_opt, beta_opt, scores = BetaA_search(A_init, max_iter=1000, res=1e-6, lr=5e-1, ld_beta=1., ld_A=1.)

# print result
score_final, Ypred_opt = metric_train(A_opt, beta_opt)
print(f'\nFinal in-sample overlap : {score_final:.5f}')

## Convergence check

In [ ]:
# Check the convergence 
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(scores)

ax.set_xlabel('iteration')
ax.set_ylabel('Score')
ax.grid(True)

plt.show()

## Visualize the Factor

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 4), sharex=True)
axes = axes.flatten()
lag_index = np.arange(1, 251)

for f, ax in enumerate(axes):
    ax.plot(lag_index, A_opt[:, f], lw=0.9, color=f'C{f}')
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.set_title(f'Factor {f + 1}', fontsize=9)
    ax.tick_params(labelsize=7)
    if f >= 5:
        ax.set_xlabel('lag (days)', fontsize=8)

fig.suptitle(r'$A$ — learning factor loadings vs. lag index (full-data training)', fontsize=11)
plt.tight_layout()
plt.show()

# Submission (Provided by QRT)

In [ ]:
def parametersTransform(A, beta, D=250, F=10):
    
    if A.shape != (D, F):
        print('A has not the good shape')
        return
    
    if beta.shape[0] != F:
        print('beta has not the good shape')
        return        
    
    output = np.hstack( (np.hstack([A.T, beta.reshape((F, 1))])).T )
    
    return output

In [ ]:
output = parametersTransform(A_opt, beta_opt)
pd.DataFrame(output).to_csv(path + 'submission2.csv')

In [ ]:
# ... and back
output_fromCsv = pd.read_csv(path + 'submission1.csv', index_col=0, sep=',').to_numpy()
A = output_fromCsv[:-10].reshape((250, 10))
beta = output_fromCsv[-10:].reshape((10))

# Additional check: the impact of the number of factors $F$

It is interesting to see that the number of factors $F$ does not necessarily influence the prediction quality. 

A reasonable guess should be that only 1-2 factors contribute significantly to the prediction.

In [ ]:
# initialization
M = X_train_reshape.values.T @ X_train_reshape.values
eigenvalues, eigenvectors = np.linalg.eigh(M)

In [ ]:
Fs = [i+1 for i in range(1, 11)]
scores_final = []
for f in Fs:
    A_init = eigenvectors[:, -f:]     # shape (250, 10), the PCA decomposition as initial guess

    # optimize
    A_opt, beta_opt, scores = BetaA_search(A_init, max_iter=1000, res=1e-6, lr=10e-1, ld_beta=0, ld_A=0, verbose=False)

    # print result
    score_final, Ypred_opt = metric_train(A_opt, beta_opt)
    print(f'\nFinal in-sample overlap : {score_final:.5f}')
    scores_final.append(score_final)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(Fs, scores_final, '-o')

ax.set_xlabel('Number of Factors')
ax.set_ylabel('Score')
ax.grid(True)

plt.show()

In [ ]:
# initialization
M = X_train_reshape.values.T @ X_train_reshape.values
eigenvalues, eigenvectors = np.linalg.eigh(M)
A_init = eigenvectors[:, -2:]     # shape (250, 10), the PCA decomposition as initial guess

# optimize
A_opt, beta_opt, scores = BetaA_search(A_init, max_iter=1000, res=1e-6, lr=5e-1, ld_beta=1., ld_A=1.)

# print result
score_final, Ypred_opt = metric_train(A_opt, beta_opt)
print(f'\nFinal in-sample overlap : {score_final:.5f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
axes = axes.flatten()
lag_index = np.arange(1, 251)

for f, ax in enumerate(axes):
    ax.plot(lag_index, A_opt[:, f], lw=0.9, color=f'C{f}')
    ax.axhline(0, color='k', lw=0.4, ls='--')
    ax.set_title(f'Factor {f + 1}', fontsize=9)
    ax.tick_params(labelsize=7)
    if f >= 5:
        ax.set_xlabel('lag (days)', fontsize=8)

fig.suptitle(r'$A$ — learning factor loadings vs. lag index (full-data training)', fontsize=11)
plt.tight_layout()
plt.show()